# Notebook 07: Integrated Testing and Evaluation

Assert representative end-to-end scenarios and verify text behaviour detection.

The event-sequence fixture is checked from `data/synthetic/integrated_event_test.csv`; it is a scenario test set, not training data.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features import detect_behaviours
from src.explain import explain_decision

scenarios = [
    ("T01 Normal transfer", {"is_new_recipient": False}, "LOW"),
    ("T02 Suspicious SMS", {"detected_behaviours": ["urgency", "threats"]}, "MEDIUM"),
    ("T03 Fake support call", {"detected_behaviours": ["authority_impersonation"]}, "MEDIUM"),
    ("T04 OTP and transaction", {"detected_behaviours": ["credential_request"], "is_new_recipient": True}, "HIGH"),
    ("T05 Account suspension", {"detected_behaviours": ["authority_impersonation", "threats"], "is_new_recipient": True}, "HIGH"),
    ("T06 Coordinated attack", {"detected_behaviours": ["authority_impersonation", "threats", "credential_request", "urgency"], "is_new_recipient": True, "new_device": True}, "CRITICAL"),
    ("T08 Large legitimate transfer", {"is_new_recipient": False, "unusual_transaction": True}, "MEDIUM"),
    ("T09 Legitimate support OTP", {"detected_behaviours": [], "is_new_recipient": False}, "LOW"),
]

failures = []
for name, indicators, expected in scenarios:
    result = explain_decision("USER_TEST", name, indicators)
    actual = result["risk_level"]
    print(name, "=>", actual, result["recommended_action"], "expected:", expected)
    if actual != expected:
        failures.append((name, expected, actual))

assert not failures, f"Scenario mismatches: {failures}"
print("All", len(scenarios), "scenario assertions passed.")

T01 Normal transfer => LOW ALLOW expected: LOW
T02 Suspicious SMS => MEDIUM WARN expected: MEDIUM
T03 Fake support call => MEDIUM WARN expected: MEDIUM
T04 OTP and transaction => HIGH ADDITIONAL_VERIFICATION expected: HIGH
T05 Account suspension => HIGH ADDITIONAL_VERIFICATION expected: HIGH
T06 Coordinated attack => CRITICAL HOLD_FOR_VERIFICATION expected: CRITICAL
T08 Large legitimate transfer => MEDIUM WARN expected: MEDIUM
T09 Legitimate support OTP => LOW ALLOW expected: LOW
All 8 scenario assertions passed.


In [2]:
legitimate = "Thank you for contacting MTN customer support. Your OTP was used for a successful transaction."
suspicious = "This is an MTN agent. Your account will be suspended. Send your OTP immediately."

legitimate_behaviours = detect_behaviours(legitimate)
suspicious_behaviours = detect_behaviours(suspicious)
print("Legitimate behaviours:", legitimate_behaviours)
print("Suspicious behaviours:", suspicious_behaviours)
assert "credential_request" not in legitimate_behaviours
assert {"authority_impersonation", "threats", "credential_request", "urgency"} <= set(suspicious_behaviours)
print("Text behaviour assertions passed.")

Legitimate behaviours: ['authority_impersonation']
Suspicious behaviours: ['authority_impersonation', 'urgency', 'threats', 'credential_request']
Text behaviour assertions passed.


In [3]:
integrated_events = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "integrated_event_test.csv")
scenario_labels = integrated_events.groupby("scenario_id")["scenario_is_fraudulent"].nunique()
assert (scenario_labels == 1).all(), "Each integrated scenario must have one consistent label."
scenario_summary = integrated_events.groupby(["scenario_id", "scenario_name"], as_index=False).agg(
    is_fraudulent=("scenario_is_fraudulent", "first"),
    includes_transaction=("is_transaction_event", "max"),
    event_count=("event_id", "count"),
)
assert scenario_summary.loc[
    scenario_summary["scenario_name"] == "suspicious_message_no_transaction", "is_fraudulent"
].item() == 1
assert scenario_summary["includes_transaction"].any()
assert (~scenario_summary["includes_transaction"].astype(bool)).any()
display(scenario_summary)

,scenario_id,scenario_name,is_fraudulent,includes_transaction,event_count
0,SCN001,account_takeover_social_engineering,1,1,3
1,SCN002,fake_refund,1,1,3
2,SCN003,legitimate_everyday_transfer,0,1,2
3,SCN004,suspicious_message_no_transaction,1,0,1
4,SCN005,legitimate_security_advice,0,1,3
